# 148 — CI/CD y pruebas para sistemas de IA

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.**
a) **Datos / bloquea**: violación de contrato de esquema (catálogo cerrado).
b) **Datos / alerta**: cambio de distribución; puede ser estacionalidad legítima — pide
revisión, no bloqueo.
c) **Modelo / bloquea**: la regresión (−0.03) excede el umbral (−0.02).
d) **Comportamiento / bloquea**: violación de invariancia semántica en un caso mínimo.
e) **Infraestructura / alerta**: anomalía operativa; no invalida el artefacto por sí
misma, pero suele anticipar un problema (datos 4× más grandes, bucle, contención).

**Ejercicio 2.** Métrica: −0.008 está dentro del umbral de bloqueo (−0.01) pero es una
caída → **alerta**. Direccional: 0.6 % < 1 % → pasa. Casos mínimos: 39/40 → **bloquea**:
los casos mínimos son contrato, un solo fallo bloquea (para eso se eligieron). Veredicto
global: **bloqueado**, con la alerta de métrica anotada en el reporte del PR.

**Ejercicio 3.** Cada implementación es internamente correcta (sus tests unitarios
pasan), pero calculan **cosas distintas** cuando el entrenamiento usa una `fecha_corte`
pasada y el serving usa `hoy`: el modelo aprende con la feature «congelada al corte» y
sirve con la feature «fresca» — skew de definición, no de código. Verificación: para una
muestra de entidades, computar la feature por ambas rutas **con la misma fecha de
referencia** y exigir igualdad exacta (o tolerancia definida); además, muestrear en
producción y comparar contra el valor offline reconstruido.

**Ejercicio 4.** Los elementos de `evidence` que describen entradas validadas son tests
de datos; los que comparan resultados contra lo esperado, tests de modelo; los que fijan
la semilla y el contrato JSON, infraestructura/reproducibilidad. `limitations` señala lo
que un CI real añadiría: umbrales acordados y bloqueo automático.


In [ ]:
result = run_lab("evaluation", seed=148)
assert result["kind"] == "evaluation"
assert result["evidence"]
show(result)


In [ ]:
delta_accuracy = -0.008
violaciones_direccionales = 0.006
casos_minimos = (39, 40)

checks = {
    "metrica (bloqueo si < -0.01)": ("bloquea" if delta_accuracy < -0.01
                                      else "alerta" if delta_accuracy < 0 else "pasa"),
    "direccional (bloqueo si > 1 %)": "bloquea" if violaciones_direccionales > 0.01 else "pasa",
    "casos minimos (contrato)": "bloquea" if casos_minimos[0] < casos_minimos[1] else "pasa",
}
for k, v in checks.items():
    print(f"{k}: {v}")
veredicto = "BLOQUEADO" if "bloquea" in checks.values() else "PASA (con alertas)" if "alerta" in checks.values() else "PASA"
print("veredicto:", veredicto)


## Reflexión

1. ¿Por qué el smoke train con 5 % de los datos es un test valioso aunque su métrica no sirva para decidir promoción? ¿Qué clase de errores atrapa?
2. Da un ejemplo concreto (de tu dominio) de test de invariancia y uno direccional, y explica qué bug de pipeline detectaría cada uno.
3. ¿Qué criterio usarías para decidir si un test nuevo entra como bloqueo o como alerta, y qué señal te diría que está mal clasificado?
